# Fine-tuning de modelo para sumarização de diálogos de atendimento ao cliente pelo Twitter

- Prepara datasets de treino, validação e teste

- Exporta datasets pré-processados

## Define constantes

In [1]:
PATH_PREPARED_DATASET_TRAIN = f'../data/interim/summarization_train.csv'
PATH_PREPARED_DATASET_VALID = f'../data/interim/summarization_valid.csv'
PATH_PREPARED_DATASET_TEST = f'../data/interim/summarization_test.csv'

PATH_DATASET_TWEETS = '../data/raw/kaggle/twcs.csv'

PATH_DATASET_SUMMARIES_TRAIN = '../data/raw/github/final_train_tweetsum.jsonl'
PATH_DATASET_SUMMARIES_VALID = '../data/raw/github/final_valid_tweetsum.jsonl'
PATH_DATASET_SUMMARIES_TEST = '../data/raw/github/final_test_tweetsum.jsonl'

## Carrega bibliotecas

In [ ]:
import os
from rich import print
import pandas as pd
from datasets import load_dataset

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## Prepara dataset

### Messages dataset - [Customer Support on Twitter (Kaggle)](https://www.kaggle.com/datasets/thoughtvector/customer-support-on-twitter)

In [3]:
df_tweets = pd.read_csv(PATH_DATASET_TWEETS)
print(df_tweets.shape)
df_tweets.head()

(2811774, 7)

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


### Annotations dataset - https://github.com/guyfe/Tweetsumm

In [4]:
df_summ_train = pd.read_json(PATH_DATASET_SUMMARIES_TRAIN, lines=True)
df_summ_valid = pd.read_json(PATH_DATASET_SUMMARIES_VALID, lines=True)
df_summ_test = pd.read_json(PATH_DATASET_SUMMARIES_TEST, lines=True)

print(f'df_summ_train.shape: {df_summ_train.shape}')
print(f'df_summ_valid.shape: {df_summ_valid.shape}')
print(f'df_summ_test.shape: {df_summ_test.shape}')

df_summ_train.head()

df_summ_train.shape: (879, 3)

df_summ_valid.shape: (110, 3)

df_summ_test.shape: (110, 3)

,conversation_id,tweet_ids_sentence_offset,annotations
0,b065262210783596c1fe79466b8f8985,"[{'tweet_id': 87076, 'sentence_offsets': ['[0,...","[{'extractive': [{'tweet_id': 87076, 'sentence..."
1,1e1d8fd4f95c984fb78687c9e946dc97,"[{'tweet_id': 607297, 'sentence_offsets': ['[0...","[{'extractive': [{'tweet_id': 607297, 'sentenc..."
2,b5773077fa55c381260390472deff4c2,"[{'tweet_id': 428556, 'sentence_offsets': ['[0...","[{'extractive': [{'tweet_id': 739293, 'sentenc..."
3,3574d6b4418cb546a90d1561bacd66a2,"[{'tweet_id': 587942, 'sentence_offsets': ['[0...","[{'extractive': [{'tweet_id': 587942, 'sentenc..."
4,f30d2bbc15e7ff32244761b38b67d160,"[{'tweet_id': 861415, 'sentence_offsets': ['[0...","[{'extractive': [{'tweet_id': 861405, 'sentenc..."


In [5]:
print(df_summ_train.iloc[0].annotations)

[
    {
        'extractive': [
            {'tweet_id': 87076, 'sentence_offset': '[0, 140]'},
            {'tweet_id': 87074, 'sentence_offset': '[41, 139]'},
            {'tweet_id': 87073, 'sentence_offset': '[0, 61]'}
        ],
        'abstractive': [
            'Customer enquired about his Iphone and Apple watch which is not showing his any steps/activity and 
health activities.',
            'Agent is asking to move to DM and look into it.'
        ]
    },
    {
        'extractive': [
            {'tweet_id': 87076, 'sentence_offset': '[0, 140]'},
            {'tweet_id': 87074, 'sentence_offset': '[41, 139]'}
        ],
        'abstractive': [
            'The customer has a problem.',
            'The agent in a very professional way tries to help the client.'
        ]
    },
    {
        'extractive': [
            {'tweet_id': 87076, 'sentence_offset': '[0, 140]'},
            {'tweet_id': 87072, 'sentence_offset': '[19, 87]'},
            {'tweet_id': 87069, 'sentence_offset': '[0, 72]'},
            {'tweet_id': 87068, 'sentence_offset': '[0, 55]'}
        ],
        'abstractive': [
            'Health and activity functions are not working with the smartwatch and phone.',
            'Asks if the customer had restarted the items, offers to take this to DM to help resolve the issue.'
        ]
    }
]

In [6]:
df_indexed = df_tweets.set_index('tweet_id')

In [7]:
def sort_tweet_text(row):
    sorted_ids = sorted(row['created_at_list'], key=lambda x: row['created_at_list'][x])
    sorted_created_at = {idx: row['created_at_list'][idx] for idx in sorted_ids}
    tweet_text_sorted = [row['tweet_text'][idx] for idx in sorted_created_at.keys()]
    return tweet_text_sorted

def calc_period(x):
    date_list = pd.to_datetime(df_indexed.loc[x, 'created_at']).tolist()
    return max(date_list) - min(date_list) 

In [ ]:
def prepare_df(df: pd.DataFrame) -> pd.DataFrame:

    df_exp = df.explode('annotations')
    df_exp['extractive'] = df_exp['annotations'].apply(lambda x: x['extractive'])
    df_exp['abstractive'] = df_exp['annotations'].apply(lambda x: x['abstractive'])

    df_exp = df_exp.dropna()
    df_exp = df_exp[~df_exp.annotations.duplicated()]

    df_exp['tweet_ids'] = df_exp['extractive'].apply(
        lambda x: sorted([y['tweet_id'] for y in x])
    )

    df_exp['tweet_text'] = df_exp['tweet_ids'].apply(
        lambda x: {k: v for k, v in zip(x, df_indexed.loc[x, 'text'].to_list())}
    )

    df_exp['created_at_list'] = df_exp['tweet_ids'].apply(
        lambda x: {k: v for k, v in zip(x, pd.to_datetime(df_indexed.loc[x, 'created_at']).tolist())}
    )

    df_exp = df_exp[~df_exp.tweet_text.duplicated()]

    df_exp['tweet_text_sorted']= df_exp[['tweet_ids', 'tweet_text', 'created_at_list']].apply(
        sort_tweet_text,
        axis=1
    )

    df_exp['human_summary'] = df_exp['abstractive'].apply(lambda x: ' '.join(x))

    df_exp.drop(columns=['tweet_ids_sentence_offset', 'annotations', 'abstractive', 'extractive', 'tweet_text'], inplace=True)
    df_exp['tweet_text_sorted'] = df_exp['tweet_text_sorted'].apply(lambda x: ' '.join(x))
    df_exp.rename(columns={'tweet_text_sorted': 'tweet_texts'}, inplace=True)

    df_exp['elapsed_time'] = df_exp['tweet_ids'].apply(
        calc_period
    )

    for idx in df_exp.conversation_id.unique()[:3]:
        df_i = df_exp[df_exp.conversation_id==idx].sort_values(by='elapsed_time').tail(1)
        df_exp.iloc[df_i.index] = df_i
    df_exp.drop_duplicates(subset=['conversation_id'], keep='first', inplace=True)

    return df_exp
    

In [9]:
df_exp_train =  prepare_df(df=df_summ_train)
df_exp_valid =  prepare_df(df=df_summ_valid)
df_exp_test =  prepare_df(df=df_summ_test)

print(df_exp_valid.iloc[0].to_dict())

{
    'conversation_id': 'caae83a2ed59e4959d814ea567980226',
    'tweet_ids': [189668, 189669, 189670, 189671],
    'created_at_list': {
        189668: Timestamp('2017-10-12 23:29:52+0000', tz='UTC'),
        189669: Timestamp('2017-10-12 23:36:18+0000', tz='UTC'),
        189670: Timestamp('2017-10-04 22:36:18+0000', tz='UTC'),
        189671: Timestamp('2017-10-04 21:32:50+0000', tz='UTC')
    },
    'tweet_texts': '@SpotifyCares hey, any explanation why the "Create similar playlist" function doesn\'t work 
anymore for me? MacBook, v1.0.64.399.g4637b02a. @160485 Hi there, the cavalry\'s here! Does logging out, restarting
your device, and logging back into Spotify help? Keep us in the loop /JI @SpotifyCares no, I tried that as well and
just reinstalled again - didn\'t help. yes, that\'s what I mean. @160485 Could you DM us your account\'s email 
address or username? We\'ll take a look backstage /MT https://t.co/ldFdZRiNAt',
    'human_summary': 'Customer is complaining that he is unable to create a new playlist in the Spotify app and 
tried reinstalling the app. Agent suggests to try logging in after restarting the device and requests  for account 
details to assist better.',
    'elapsed_time': Timedelta('8 days 02:03:28')
}

In [10]:
print(f'df_exp_train.shape: {df_exp_train.shape}')
print(f'df_exp_valid.shape: {df_exp_valid.shape}')
print(f'df_exp_test.shape: {df_exp_test.shape}')

df_exp_train.shape: (866, 6)

df_exp_valid.shape: (110, 6)

df_exp_test.shape: (109, 6)

## Exporta datasets

In [11]:
df_exp_train.to_csv(PATH_PREPARED_DATASET_TRAIN, index=None)
df_exp_test.to_csv(PATH_PREPARED_DATASET_TEST, index=None)
df_exp_valid.to_csv(PATH_PREPARED_DATASET_VALID, index=None)